Welcome to the module introducing how to access Wharton Research Data Services (WRDS) using Python.

Access to high-quality financial and economic data is fundamental for empirical research in AccFin. The Wharton Research Data Services (WRDS) platform, which provides a gateway to a wide range of leading databases such as Compustat, CRSP and IBES, is the most commonly used data platform in our empirical research. In this session, we will learn how to use the `wrds` Python package to connect programmatically to WRDS, query structured datasets, and efficiently download and manage data for analysis. The class emphasizes both the technical skills required for accessing large-scale databases and the research practices necessary to ensure data integrity, reproducibility, and efficiency in academic and applied settings.

**Learning Outcomes**

By the end of this class, you will be able to:

- Understand the role of WRDS as a central platform for accessing accounting and financial datasets.

- Establish a secure connection to WRDS using the wrds Python package.

- Formulate and execute SQL queries within Python to retrieve specific datasets from WRDS.

- Download and manage large datasets, including filtering, merging, and exporting for further analysis.

- Integrate WRDS data with Python workflows, preparing it for use in packages such as `pandas` and `statsmodels`.

- Apply best practices for reproducible research, including documenting queries and saving data pipelines.

### 0. Installation and Setup
```bash
pip install wrds
```
For documentation, see: https://github.com/wharton/wrds

Offical tutorial: https://wrds-www.wharton.upenn.edu/pages/support/programming-wrds/programming-python/querying-wrds-data-python/

In [ ]:
import wrds
import pandas as pd

### 1. Connecting to WRDS Python Module

In [ ]:
db = wrds.Connection(wrds_username='leonardl')

In [ ]:
# Create a .pgpass file for password authentication
db.create_pgpass_file()

### 2. Querying the Dataset Structure (Metadata)

#### 2.1. List the libraries available at WRDS

In [ ]:
db.list_libraries()

#### 2.2. List the datasets (tables) in a given Library 

In [ ]:
db.list_tables(library="comp")

In [ ]:
db.list_tables(library='crsp')

In [ ]:
#For example, we can list all tables of Option Price in the OptionMetrics library
[i for i in db.list_tables(library='optionm') if 'opprcd' in i]

### 2.3. List the column headers (variables) within a given dataset:

In [ ]:
db.describe_table(library="crsp", table="dsf")

### 3. Querying WRDS Data Using `raw_sql()`
Executes a SQL query against the specified library and dataset, allowing for highly-granular data queries.

parameters:

- sql - the SQL string to query
- date_cols - a list or dict of column names to parse as date (optional)

In [ ]:
comp = db.raw_sql(
        '''
        SELECT gvkey, datadate, at FROM comp.funda 
            WHERE fyear = 2020 
                AND at IS NOT NULL
        ''', date_cols=['datadate'])

In [ ]:
start_year = 2010
end_year = 2023
query = f"""
        SELECT a.*, b.permno 
		FROM 
			(
            SELECT gvkey, datadate, fyear, at, ceq, sale, ni,
					(
					CASE WHEN sale IS NULL THEN 1 ELSE 0 END +
					CASE WHEN capx IS NULL THEN 1 ELSE 0 END +
					CASE WHEN xrd IS NULL THEN 1 ELSE 0 END +
					CASE WHEN ni IS NULL THEN 1 ELSE 0 END + 
					CASE WHEN dltt IS NULL THEN 1 ELSE 0 END + 
					CASE WHEN ceq IS NULL THEN 1 ELSE 0 END + 
					CASE WHEN oancf IS NULL THEN 1 ELSE 0 END + 
					CASE WHEN prcc_f IS NULL THEN 1 ELSE 0 END + 
					CASE WHEN csho IS NULL THEN 1 ELSE 0 END + 
					CASE WHEN au IS NULL THEN 1 ELSE 0 END
					) AS miss_count 
				FROM comp.funda 
				WHERE fyear BETWEEN {start_year} AND {end_year}
                	AND fyear IS NOT NULL 
					AND indfmt='INDL' AND datafmt='STD' AND popsrc='D' AND consol='C' 
					AND at IS NOT NULL 
                    AND at > 0 
            ) AS a, 
            (
            SELECT gvkey, lpermno AS permno, linkdt, linkenddt
				FROM crsp.ccmxpf_linktable
				WHERE linktype in ('LU','LC','LS')
            ) AS b
		WHERE a.gvkey = b.gvkey
			AND (a.datadate >= b.linkdt OR b.linkdt IS NULL)
			AND (a.datadate <= b.linkenddt OR b.linkenddt IS NULL)
		ORDER BY a.gvkey, fyear, miss_count, datadate desc, permno
        """

In [ ]:
with wrds.Connection(wrds_username='leonardl') as db:
    CCM = db.raw_sql(query, date_cols=['datadate']).drop_duplicates(subset=['gvkey', 'fyear']).drop(columns=['miss_count'])

### Save the file to Stata file or csv file

In [ ]:
CCM.to_stata('Comp_with_permno.dta', write_index=False, convert_dates={'datadate': 'td'})

In [ ]:
CCM.to_csv('Comp_with_permno.csv', sep='\t', index=False)